# Quicksort Pairwise Swap Benchmark

Run the benchmark executable first, for example:

```bash
./build-local/benchmark_quicksort_pairwise_swap --output build-local/quicksort_pairwise_swap_benchmark.tsv
```

For the low-GB entries, use a smaller trial count unless you enjoy waiting:

```bash
./build-local/benchmark_quicksort_pairwise_swap --include-low-gb --trials 1 --output build-local/quicksort_pairwise_swap_benchmark.tsv
```

Inspect the selected byte-sized inputs without running the benchmark:

```bash
./build-local/benchmark_quicksort_pairwise_swap --include-low-gb --list-sizes
```

The benchmark verifies each pairwise-swap result by exact comparison with the corresponding `std::sort` output before writing the TSV row pair. The notebook expects the TSV emitted by `benchmark_quicksort_pairwise_swap`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd

def format_bytes(value, _position=None):
    value = float(value)
    for suffix in ['B', 'KiB', 'MiB', 'GiB']:
        if value < 1024 or suffix == 'GiB':
            return f'{value:g}{suffix}'
        value /= 1024

candidates = [
    Path('build-local-release/benchmark_quicksort_pairwise_swap_32bit.tsv')
    # Path('quicksort_pairwise_swap_benchmark.tsv'),
    # Path('tslctmp/quicksort_pairwise_swap_benchmark.tsv'),
    # Path('build-local/quicksort_pairwise_swap_benchmark.tsv'),
    # Path('build/quicksort_pairwise_swap_benchmark.tsv'),
    # Path('test-sort/build-local/quicksort_pairwise_swap_benchmark.tsv'),
    # Path('test-sort/build/quicksort_pairwise_swap_benchmark.tsv'),
    # Path('../tslctmp/quicksort_pairwise_swap_benchmark.tsv'),
]
data_path = next((path for path in candidates if path.exists()), candidates[0])
print(f'Loading {data_path}')
df = pd.read_csv(data_path, sep='\t')
df.head()

In [ ]:
summary = (
    df.groupby(['distribution', 'size', 'bytes', 'size_label', 'algorithm'], as_index=False)
      .agg(
          median_ns_per_element=('ns_per_element', 'median'),
          median_mib_per_second=('mib_per_second', 'median'),
          median_elapsed_ns=('elapsed_ns', 'median'),
          trials=('trial', 'nunique'),
      )
)
summary.head()

In [ ]:
distributions = list(summary['distribution'].drop_duplicates())
fig, axes = plt.subplots(len(distributions), 1, figsize=(9, 2.8 * len(distributions)), sharex=True)
if len(distributions) == 1:
    axes = [axes]

for axis, distribution in zip(axes, distributions):
    subset = summary[summary['distribution'] == distribution]
    for algorithm, group in subset.groupby('algorithm'):
        group = group.sort_values('bytes')
        axis.plot(group['bytes'], group['median_ns_per_element'], marker='o', label=algorithm)
    axis.set_xscale('log', base=2)
    axis.xaxis.set_major_formatter(mticker.FuncFormatter(format_bytes))
    axis.set_ylabel('ns / element')
    axis.set_title(distribution)
    axis.grid(True, which='both', axis='both', alpha=0.25)
    axis.legend()

axes[-1].set_xlabel('input footprint')
fig.tight_layout()

In [ ]:
pivot = summary.pivot_table(
    index=['distribution', 'size', 'bytes', 'size_label'],
    columns='algorithm',
    values='median_ns_per_element',
).reset_index()
pivot['speedup_vs_std_sort'] = pivot['std_sort'] / pivot['tsl_pairwise_swap']

fig, axis = plt.subplots(figsize=(9, 5))
for distribution, group in pivot.groupby('distribution'):
    group = group.sort_values('bytes')
    axis.plot(group['bytes'], group['speedup_vs_std_sort'], marker='o', label=distribution)

axis.axhline(1.0, color='black', linewidth=1, linestyle='--')
axis.set_xscale('log', base=2)
axis.xaxis.set_major_formatter(mticker.FuncFormatter(format_bytes))
axis.set_xlabel('input footprint')
axis.set_ylabel('std_sort ns/elem / tsl_pairwise_swap ns/elem')
axis.set_title('Speedup vs std_sort (higher is better for tsl_pairwise_swap)')
axis.grid(True, which='both', axis='both', alpha=0.25)
axis.legend()
fig.tight_layout()

In [ ]:
if 'vqsort' not in pivot.columns:
    print('No vqsort rows found in the benchmark TSV.')
else:
    pivot['slowdown_vs_vqsort'] = pivot['tsl_pairwise_swap'] / pivot['vqsort']

    fig, axis = plt.subplots(figsize=(9, 5))
    for distribution, group in pivot.groupby('distribution'):
        group = group.sort_values('bytes')
        axis.plot(group['bytes'], group['slowdown_vs_vqsort'], marker='o', label=distribution)

    axis.axhline(1.0, color='black', linewidth=1, linestyle='--')
    axis.set_xscale('log', base=2)
    axis.xaxis.set_major_formatter(mticker.FuncFormatter(format_bytes))
    axis.set_xlabel('input footprint')
    axis.set_ylabel('tsl_pairwise_swap ns/elem / vqsort ns/elem')
    axis.set_title('Slowdown vs vqsort (lower is better for tsl_pairwise_swap)')
    axis.grid(True, which='both', axis='both', alpha=0.25)
    axis.legend()
    fig.tight_layout()

In [ ]:
pivot.sort_values(['distribution', 'bytes'])